# Notebook 1 — M/M/1 Single-Server Queue

**Series:** DES Playground · Discrete Event Simulation in SimPy  
**Prerequisites:** None  
**What you'll learn:**
- Build a single-server queue in SimPy from scratch
- Understand *warm-up* and *observation windows*
- Run multiple replications and compute 95 % confidence intervals
- Verify results against the M/M/1 closed-form formulas
- See why queue length *explodes* as utilisation approaches 100 %

---

## 1 · Background — What Is an M/M/1 Queue?

Kendall notation **M/M/1** means:

| Symbol | Meaning |
|--------|--------|
| First M | **M**arkovian (exponential) inter-arrival times |
| Second M | **M**arkovian (exponential) service times |
| 1 | **1** server |

With arrival rate **λ** and service rate **μ**, the utilisation is **ρ = λ/μ**.  
The queue is *stable* only when **ρ < 1**.  For a stable M/M/1 system the
steady-state formulas are:

```
Lq = ρ² / (1 − ρ)          mean number waiting in queue
L  = ρ  / (1 − ρ)          mean number in system
Wq = Lq / λ                mean wait in queue   (Little's Law)
W  = L  / λ                mean sojourn time    (Little's Law)
```

These are the **benchmarks** we will verify with simulation.

## 2 · Setup

In [ ]:
# Run this cell once to install SimPy (already present on most Colab runtimes)
!pip install simpy --quiet

In [ ]:
import simpy
import random
import math
import statistics
from dataclasses import dataclass, field
from typing import List, Dict
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('SimPy version:', simpy.__version__)

## 3 · Simulation Framework

We build a reusable `Station` class that tracks **time-weighted** queue
statistics.  The key idea: instead of sampling the queue at fixed intervals
(biased), we integrate the area under the queue-length curve by multiplying
each count by the duration it was held.

```
   Lq = (∫ queue_length dt) / observation_window
   ρ̂  = (∫ busy_servers  dt) / (observation_window × capacity)
```

We also distinguish the **warm-up period** (transient, excluded from metrics)
from the **observation window** (steady-state, used for metrics).

In [ ]:
# ── T-distribution critical values for 95 % two-sided CI ─────────────────────
T_TABLE = {
    1:12.706, 2:4.303, 3:3.182, 4:2.776, 5:2.571,
    6:2.447,  7:2.365, 8:2.306, 9:2.262, 10:2.228,
    15:2.131, 20:2.086, 25:2.060, 30:2.042
}

def t_crit(df: int) -> float:
    if df in T_TABLE:
        return T_TABLE[df]
    if df > 30:
        return 1.960
    keys = sorted(T_TABLE)
    for i in range(len(keys) - 1):
        lo, hi = keys[i], keys[i+1]
        if lo < df < hi:
            return T_TABLE[lo] + (T_TABLE[hi] - T_TABLE[lo]) * (df - lo) / (hi - lo)
    return 2.0

def ci(values: List[float]) -> Dict:
    """95 % confidence interval via t-distribution."""
    n = len(values)
    if n < 2:
        return dict(mean=values[0] if values else 0.0, half=0.0, lo=values[0] if values else 0.0, hi=values[0] if values else 0.0)
    mu = statistics.mean(values)
    s  = statistics.stdev(values)
    h  = t_crit(n - 1) * s / math.sqrt(n)
    return dict(mean=mu, std=s, half=h, lo=mu - h, hi=mu + h, n=n)


# ── Station: single resource with time-weighted metrics ───────────────────────
class Station:
    """
    Wraps a SimPy Resource and tracks:
      - queue-area and busy-area integrals (for time-weighted Lq, rho)
      - per-entity wait and sojourn times (for entity-cohort Wq, W)
    """
    def __init__(self, env: simpy.Environment, name: str, capacity: int = 1):
        self.env      = env
        self.name     = name
        self.resource = simpy.Resource(env, capacity=capacity)
        self.capacity = capacity
        # running integrals
        self._last_t     = 0.0
        self._queue_area = 0.0
        self._busy_area  = 0.0
        # per-entity records (only entities that arrived >= warmup)
        self.wait_times:    List[float] = []
        self.sojourn_times: List[float] = []
        self.arrivals   = 0
        self.departures = 0

    def _integrate(self, t: float):
        """Advance the integrals to time t."""
        dt = t - self._last_t
        self._queue_area += len(self.resource.queue) * dt
        self._busy_area  += self.resource.count * dt
        self._last_t = t

    # Call at event boundaries (arrival, service start, service end)
    def tick(self):
        self._integrate(self.env.now)

    def avg_queue_length(self, warmup: float, horizon: float) -> float:
        self._integrate(horizon)
        obs = max(horizon - warmup, 1.0)
        return self._queue_area / obs

    def avg_utilization(self, warmup: float, horizon: float) -> float:
        self._integrate(horizon)
        obs = max(horizon - warmup, 1.0)
        return (self._busy_area / obs) / self.capacity

    def avg_wait(self) -> float:
        return statistics.mean(self.wait_times) if self.wait_times else 0.0

    def avg_sojourn(self) -> float:
        return statistics.mean(self.sojourn_times) if self.sojourn_times else 0.0

print('Station class ready.')

## 4 · The M/M/1 Simulation

The simulation loop:

1. A **source** generates entities at exponential inter-arrival times.
2. Each entity **requests** the server (waits in queue if busy).
3. After being granted the server, it holds it for an exponential **service time**.
4. It **releases** the server and departs.

Entities that arrive **after the warm-up** period have their wait and sojourn
times recorded — this removes cold-start bias from the metrics.

In [ ]:
def run_mm1(
    arrival_rate:  float,   # λ — mean arrivals per time unit
    service_rate:  float,   # μ — mean services per time unit
    horizon:       float = 10_000,
    warmup:        float = 500,
    seed:          int   = 42,
) -> Dict:
    """Run one M/M/1 replication and return a metrics dict."""
    rng = random.Random(seed)
    env = simpy.Environment()
    srv = Station(env, 'Server', capacity=1)

    # Queue-length time-series sampled every Δt for plotting
    ts_t:  List[float] = []
    ts_q:  List[int]   = []

    def entity(arrival_t: float):
        """Lifecycle of one entity."""
        track = arrival_t >= warmup   # only record post-warmup entities
        srv.tick()                    # update integrals before queue changes
        srv.arrivals += 1

        queue_enter_t = env.now
        with srv.resource.request() as req:
            yield req                 # wait until server is free
            srv.tick()
            wait = env.now - queue_enter_t

            # Service
            svc_time = rng.expovariate(service_rate)
            yield env.timeout(svc_time)
            srv.tick()

        # Departure
        srv.departures += 1
        if track:
            srv.wait_times.append(wait)
            srv.sojourn_times.append(wait + svc_time)

    def source():
        """Generate entities at exponential inter-arrival times."""
        while True:
            yield env.timeout(rng.expovariate(arrival_rate))
            env.process(entity(env.now))

    def sampler(interval=50):
        """Record queue length periodically."""
        while True:
            ts_t.append(env.now)
            ts_q.append(len(srv.resource.queue))
            yield env.timeout(interval)

    env.process(source())
    env.process(sampler())
    env.run(until=horizon)

    rho = arrival_rate / service_rate
    return {
        'lq':  srv.avg_queue_length(warmup, horizon),
        'rho': srv.avg_utilization(warmup, horizon),
        'wq':  srv.avg_wait(),
        'w':   srv.avg_sojourn(),
        'throughput': srv.departures / (horizon - warmup),
        'ts_t': ts_t,
        'ts_q': ts_q,
        'rho_theory': rho,
    }

print('run_mm1() defined.')

## 5 · Run a Single Simulation

Let's start with **λ = 0.9, μ = 1.0** → **ρ = 0.9** (heavily loaded).

In [ ]:
LAMBDA = 0.9   # arrival rate
MU    = 1.0    # service rate
RHO   = LAMBDA / MU

res = run_mm1(arrival_rate=LAMBDA, service_rate=MU, horizon=20_000, warmup=1_000, seed=42)

# M/M/1 theory
lq_theory = RHO**2 / (1 - RHO)
l_theory  = RHO    / (1 - RHO)
wq_theory = lq_theory / LAMBDA
w_theory  = l_theory  / LAMBDA

print(f'ρ (target) = {RHO:.2f}')
print()
print(f'{"Metric":<18} {"Simulation":>12} {"Theory":>12} {"Relative err":>14}')
print('-' * 60)
for metric, sim_val, theory_val in [
    ('Utilisation ρ',  res['rho'], RHO),
    ('Avg queue Lq',   res['lq'],  lq_theory),
    ('Avg wait   Wq',  res['wq'],  wq_theory),
    ('Avg sojourn W',  res['w'],   w_theory),
]:
    err = abs(sim_val - theory_val) / max(theory_val, 1e-9) * 100
    print(f'{metric:<18} {sim_val:>12.4f} {theory_val:>12.4f} {err:>13.1f}%')

### Queue Length Over Time

The plot below shows the warm-up period (grey) and the observation window (blue).
Notice how the queue fluctuates around the theoretical mean during steady state.

In [ ]:
WARMUP = 1_000

fig, ax = plt.subplots(figsize=(12, 4))

t = np.array(res['ts_t'])
q = np.array(res['ts_q'])

warm_mask = t < WARMUP
obs_mask  = t >= WARMUP

ax.fill_between(t[warm_mask], q[warm_mask], alpha=0.2, color='orange', label='Warm-up (excluded)')
ax.fill_between(t[obs_mask],  q[obs_mask],  alpha=0.3, color='steelblue', label='Observation window')
ax.plot(t, q, lw=0.6, color='steelblue', alpha=0.7)
ax.axvline(WARMUP, color='orange', ls='--', lw=1.5, label=f'Warm-up boundary t={WARMUP}')
ax.axhline(lq_theory, color='red', ls='--', lw=1.5, label=f'Theory Lq = {lq_theory:.2f}')

ax.set_xlabel('Simulation time')
ax.set_ylabel('Queue length')
ax.set_title(f'M/M/1 queue — λ={LAMBDA}, μ={MU}, ρ={RHO:.2f}')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 6 · Multiple Replications & Confidence Intervals

A single run gives one point estimate.  To quantify uncertainty we run
**10 independent replications** (each with a different seed) and compute
a 95 % confidence interval using the t-distribution.

The DES Playground uses seed **s + i × 1000** for replication *i*.

In [ ]:
BASE_SEED = 42
N_REPS    = 10

rep_lq, rep_rho, rep_wq, rep_w = [], [], [], []

for i in range(N_REPS):
    r = run_mm1(LAMBDA, MU, horizon=20_000, warmup=1_000, seed=BASE_SEED + i*1000)
    rep_lq.append(r['lq'])
    rep_rho.append(r['rho'])
    rep_wq.append(r['wq'])
    rep_w.append(r['w'])

print(f'{"Metric":<16} {"Mean":>8} {"±95%CI":>8} {"Theory":>8}')
print('-' * 48)
for label, vals, theory in [
    ('Lq',  rep_lq,  lq_theory),
    ('ρ̂',  rep_rho, RHO),
    ('Wq',  rep_wq,  wq_theory),
    ('W',   rep_w,   w_theory),
]:
    c = ci(vals)
    print(f'{label:<16} {c["mean"]:>8.3f} {c["half"]:>8.3f} {theory:>8.3f}')

## 7 · Queue Explosion — Lq vs ρ

This is the most important insight in queueing theory:
**Lq grows nonlinearly** as ρ → 1.  Moving from 80 % to 90 % utilisation
doesn't just add 10 % more queue — it *triples* it.

We sweep ρ from 0.1 to 0.95 and overlay the theory curve.

In [ ]:
rho_vals   = np.linspace(0.1, 0.97, 25)
lq_theory_curve = rho_vals**2 / (1 - rho_vals)

sim_means, sim_cilo, sim_cihi = [], [], []
MU_FIXED = 1.0

for rho in rho_vals:
    lam = rho * MU_FIXED
    runs = [run_mm1(lam, MU_FIXED, horizon=15_000, warmup=500, seed=42+k*1000)['lq']
            for k in range(5)]
    c = ci(runs)
    sim_means.append(c['mean'])
    sim_cilo.append(c['lo'])
    sim_cihi.append(c['hi'])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rho_vals, lq_theory_curve, 'r-', lw=2.5, label='Theory: Lq = ρ²/(1−ρ)')
ax.plot(rho_vals, sim_means, 'o', color='steelblue', ms=5, label='Simulation mean (5 reps)')
ax.fill_between(rho_vals, sim_cilo, sim_cihi, alpha=0.25, color='steelblue', label='95 % CI')

ax.set_xlabel('Utilisation ρ')
ax.set_ylabel('Mean queue length Lq')
ax.set_title('M/M/1 — Queue length vs Utilisation')
ax.set_xlim(0, 1.0)
ax.set_ylim(0, None)
ax.axvspan(0.9, 1.0, alpha=0.07, color='red', label='Danger zone ρ > 0.9')
ax.legend()
plt.tight_layout()
plt.show()

print(f'At ρ=0.5 → Lq≈{0.5**2/(1-0.5):.1f}  |  ρ=0.8 → Lq≈{0.8**2/(1-0.8):.1f}  |  ρ=0.9 → Lq≈{0.9**2/(1-0.9):.1f}  |  ρ=0.95 → Lq≈{0.95**2/(1-0.95):.1f}')

## 8 · Little's Law Verification

**Little's Law** states that for any stable system:

```
L = λ · W    (mean number in system = arrival rate × mean sojourn time)
Lq = λ · Wq  (same law applied to the queue only)
```

This is a **universal** result — it holds regardless of arrival distribution,
service distribution, or number of servers.  Let's verify it.

In [ ]:
res_check = run_mm1(LAMBDA, MU, horizon=50_000, warmup=1_000, seed=99)

L_measured  = res_check['lq'] + res_check['rho']  # Lq + rho (= L for M/M/1)
L_littles   = LAMBDA * res_check['w']

Lq_measured = res_check['lq']
Lq_littles  = LAMBDA * res_check['wq']

print('Little\'s Law: L = λ·W')
print(f'  L (time-weighted):  {L_measured:.4f}')
print(f'  λ · W (entity):     {L_littles:.4f}')
print(f'  Relative error:     {abs(L_measured-L_littles)/L_measured*100:.2f}%')
print()
print('Little\'s Law: Lq = λ·Wq')
print(f'  Lq (time-weighted): {Lq_measured:.4f}')
print(f'  λ · Wq (entity):    {Lq_littles:.4f}')
print(f'  Relative error:     {abs(Lq_measured-Lq_littles)/max(Lq_measured,1e-9)*100:.2f}%')

print('\n✓ Both should be < 2 % for a long run — confirming the simulator is correct.')

## 9 · Summary & Key Takeaways

| Concept | Takeaway |
|---------|----------|
| **Warm-up** | Always exclude the transient phase; cold-start inflates Wq estimates |
| **Replications** | Multiple seeds give confidence intervals, not just point estimates |
| **Nonlinearity** | Queue length is nonlinear in ρ — moving 80→90 % *triples* the queue |
| **Little's Law** | L = λW links time-weighted and entity-cohort metrics universally |
| **Stability** | ρ ≥ 1 means the queue grows without bound — there is no steady state |

**Next:** [Notebook 2](02_mmc_multi_server.ipynb) — M/M/c: add more servers and see when it helps.